# DFX4ML Multi-Region Multi-RM Pipeline Test

End-to-end test for a **2-region × 2-RM** DFX4ML system.

The system uses **partial reconfiguration** to dynamically swap hardware accelerator kernels
(Reconfigurable Modules, RMs) across two independent PR regions at runtime.

### Pipeline sequence
```
DMA input
  → region 0 / RM 0   (+0)
  → region 1 / RM 0   (+2)
  → region 0 / RM 1   (+1)   ← region 0 is reconfigured here
  → region 1 / RM 1   (+3)   ← region 1 is reconfigured here
→ DMA output   (expected: input + 6)
```

Each RM is a `Stream_Single_S2M` that adds its compile-time constant
`vm_id * amount_region + rm_id` to every output word.

### Slot table layout (5 slots, cyclic after slot 0)
| Slot | Recon          | Exec    | Data flow              |
|------|----------------|---------|------------------------|
| 0    | r0 → rm0       | —       | none (init only)       |
| 1    | r1 → rm0       | r0/rm0  | DMA in → MGS1          |
| 2    | r0 → rm1       | r1/rm0  | MGS1 → MGS2            |
| 3    | r1 → rm1       | r0/rm1  | MGS2 → MGS1            |
| 4    | r0 → rm0       | r1/rm1  | MGS1 → DMA out         |

Slot 0 runs once (init); slots 1–4 repeat cyclically (slot 4 → slot 1).

> **Hardware Setup Required:** This notebook must run on a PYNQ board with the exported bitstreams in `hw/`.

## Step 1 — Project Configuration

In [ ]:
import os
import time
import asyncio
import numpy as np
from pynq import Overlay, allocate, Interrupt

# do not remove this import
import driver.cap           as cap
import driver.mem_alloc     as dataAlloc
import driver.dfx_unified   as dfx_unified
import driver.dfx_mgs_debug as dfx_mgs_debug

# ── paths ─────────────────────────────────────────────────────────────────
PRJ_DIR    = os.getcwd()
PRJ_HW_DIR = os.path.join(PRJ_DIR, 'hw')
PRJ_TC_DIR = os.path.join(PRJ_DIR, 'data')

# ── system configuration ──────────────────────────────────────────────────
# Must match the values used when building.
NUM_PR_REGION      = 2   # number of independent reconfigurable regions
NUM_RM_PER_REGION  = 2   # number of RM variants per region
TOTAL_RM           = NUM_PR_REGION * NUM_RM_PER_REGION  # 4
LAST_SESSION       = 4   # last slot index used (slots 0–4, 5 slots total)

# ── vs_rm_recon_sel / vs_rm_exec_sel bit encoding ─────────────────────────
# Bits are packed as: [r0_rm0, r0_rm1, r1_rm0, r1_rm1]
#   bit 0 = region 0, rm 0
#   bit 1 = region 0, rm 1
#   bit 2 = region 1, rm 0
#   bit 3 = region 1, rm 1
RM_SEL = {
    (0, 0): 0b0001,   # r0_rm0
    (0, 1): 0b0010,   # r0_rm1
    (1, 0): 0b0100,   # r1_rm0
    (1, 1): 0b1000,   # r1_rm1
}

MGS_SEL_DMA  = 0b00001
MGS_SEL_MGS1 = 0b00010
MGS_SEL_MGS2 = 0b00100
MGS_SEL_MGS3 = 0b01000
MGS_SEL_MGS4 = 0b10000

# ── bitstream names ───────────────────────────────────────────────────────
FULL_BS_NAME = 'system.bin'

# Partial bitstream for region r, RM m  →  hw/region_{r}_rm_{m}.bin
PAR_BS = {
    (r, m): os.path.join(PRJ_HW_DIR, f'region_{r}_rm_{m}.bin')
    for r in range(NUM_PR_REGION)
    for m in range(NUM_RM_PER_REGION)
}

# ── data parameters ───────────────────────────────────────────────────────
INPUT_DATA_NAME = 'x_input.npy'
AMT_QUERY       = 900
INPUT_SHAPE     = (AMT_QUERY, 8,8,1)
OUTPUT_SHAPE    = (AMT_QUERY, 4)

# ── physical address map (must match board_build.tcl / HWH) ──────────────
DMA_PHY_ADDR     = 0xA003_0000
PR_CTRL_PHY_BASE = 0xA005_0000   # region 0; region r → base + r * stride
PR_CTRL_STRIDE   = 0x0001_0000

pr_phy_addrs = [PR_CTRL_PHY_BASE + r * PR_CTRL_STRIDE for r in range(NUM_PR_REGION)]
print('PR ctrl physical addresses:', [hex(a) for a in pr_phy_addrs])
print(f'TOTAL_RM = {TOTAL_RM},  LAST_SESSION = {LAST_SESSION}')

In [ ]:
# Generate synthetic input data and save to disk
# input_x = (np.arange(AMT_QUERY, dtype=np.int32) + 48).reshape(INPUT_SHAPE)
# np.save(os.path.join(PRJ_TC_DIR, INPUT_DATA_NAME), input_x)

## Step 2 — Load the Full Bitstream (Static Overlay)

In [ ]:
cap.change_pl_config_mode('pcap', True, '')
overlay = Overlay(os.path.join(PRJ_HW_DIR, FULL_BS_NAME))
print('Overlay loaded.')

## Step 3 — Register Interrupt

In [ ]:
overlay.interrupt_pins
my_interrupt = Interrupt('dfx_unified_0/dfx_intr')

## Step 4 — Access IP Sub-blocks

| Handle | Role |
|---|---|
| `dfx_mng`  | DFX Manager — orchestrates DMA → compute → reconfig sequence |
| `dfx_ctrl` | DFX Controller — drives ICAP for partial reconfiguration |
| `dfx_dma`  | AXI DMA — moves data between DDR and PR regions |
| `dfx_man`  | PR Decoupler / Reset manager (single instance, all regions) |
| `pr_ctrl[r]` | Per-region HLS kernel control (AP_CTRL interface) |

In [ ]:
dfx_ip  = overlay.dfx_unified_0

dfx_mng  = dfx_ip.dfx_mng
dfx_ctrl = dfx_ip.dfx_ctrl
dfx_dma  = dfx_ip.dfx_dma
dfx_man  = dfx_ip.dfx_man

print(f'NUM_PR_REGION = {dfx_ip.NUM_PR_REGION}')
print(f'LIM_AMT_SLOT  = {1 << dfx_ip.SLOT_INDEX_WIDTH}')
assert dfx_ip.NUM_PR_REGION == NUM_PR_REGION, 'Region count mismatch'
assert (1 << dfx_ip.SLOT_INDEX_WIDTH) > LAST_SESSION, 'Slot table too small for LAST_SESSION'

## Step 5 — Configure DFX Controller

In [ ]:
DFX_CONFIG_FILE = 'dfx_ctrl_con.txt'
dfx_ctrl.config(os.path.join(PRJ_HW_DIR, DFX_CONFIG_FILE))
print('dfx_ctrl configured. BLS_REGID =', dfx_ctrl.BLS_REGID)

# Switch PL config interface to ICAP for runtime partial reconfiguration.
cap.change_pl_config_mode('icap', True, '')

## Step 6 — Initial System Reset

In [ ]:
dfx_mng.shutdown_engine()
for r in range(NUM_PR_REGION):
    dfx_ctrl.shutdown_engine(r)
    dfx_ctrl.print_status(r)

## Step 7 — Initialise DFX Manager (Bank 0 Metadata)

In [ ]:
print('------ before init ------')
dfx_mng.print_debug()

print('------ init Bank 0 metadata ------')
dfx_mng.set_last_session(LAST_SESSION)   # slots 0 … 7
dfx_mng.set_dma_ip_addr(DMA_PHY_ADDR)
dfx_mng.set_pr_ip_addr(pr_phy_addrs[0])  # region-0 pr_ctrl; HW uses stride for others
dfx_mng.set_amt_query(AMT_QUERY)
dfx_mng.set_amt_query_per_iter(AMT_QUERY)
dfx_mng.set_intr_ena(1)

In [ ]:
# Allocate input / output / intermediate CMA buffers
inputX = np.load(os.path.join(PRJ_TC_DIR, INPUT_DATA_NAME))
assert inputX.shape == INPUT_SHAPE, f'Shape mismatch: {inputX.shape} vs {INPUT_SHAPE}'

buf_in,   buf_in_phya,   buf_in_sz   = dataAlloc.alloc_data_uint(
    alloc_shape=INPUT_SHAPE,  alloc_type=np.float32, input_x=inputX)
buf_out,  buf_out_phya,  buf_out_sz  = dataAlloc.alloc_data_uint(
    alloc_shape=OUTPUT_SHAPE, alloc_type=np.float32)

buf_in.flush()
print('CMA buffers allocated.')
print(f'  input  @ {hex(buf_in_phya)},  size={hex(buf_in_sz)}')
print(f'  output @ {hex(buf_out_phya)},  size={hex(buf_out_sz)}')

In [ ]:
# ── Configure slot table (Bank 1) ─────────────────────────────────────────
# Slot 0 — Init: recon region 0 → rm 0
dfx_mng.set_whole_slot(0, [
    0            , 0,          # src (none)
    0            , 0,          # dst (none)
    0            , 0,          # profile counters
    RM_SEL[(0,0)], 0,          # recon r0_rm0 / no exec
    0            , 0,          # load mask/ store mask
    0,                         # mas
    1,                         # next → slot 1
])

dfx_mng.set_whole_slot(1, [
    buf_in_phya   , buf_in_sz                   , # src (none)
    0             , 0                           , # dst (none)
    0             , 0                           , # profile counters
    RM_SEL[(1,0)] , RM_SEL[(0,0)]               , # recon r0_rm0 / no exec
    MGS_SEL_DMA   , MGS_SEL_MGS1 | MGS_SEL_MGS2 , # load mask/ store mask
    0             ,                               # mas
    2             ,                               # next → slot 2
])

dfx_mng.set_whole_slot(2, [
    0                            , 0                           , # src (none)
    0                            , 0                           , # dst (none)
    0                            , 0                           , # profile counters
    RM_SEL[(0,1)]                , RM_SEL[(1,0)]               , # recon r0_rm0 / no exec
    MGS_SEL_MGS1 | MGS_SEL_MGS2  , MGS_SEL_MGS3 | MGS_SEL_MGS4 , # load mask/ store mask
    0                            ,                               # mas
    3,                                                           # next → slot 3
])

dfx_mng.set_whole_slot(3, [
    0                            , 0             , # src (none)
    0                            , 0             , # dst (none)
    0                            , 0             , # profile counters
    RM_SEL[(1,1)]                , RM_SEL[(0,1)] , # recon r0_rm0 / no exec
    MGS_SEL_MGS3 | MGS_SEL_MGS4  , MGS_SEL_MGS1  , # load mask/ store mask
    0             ,                # mas
    4,                             # next → slot 4
])

dfx_mng.set_whole_slot(4, [
    0                           , 0            , # src (none)
    buf_out_phya                , buf_out_sz   , # dst (none)
    0                           , 0            , # profile counters
    RM_SEL[(0,0)]               , RM_SEL[(1,1)], # recon r0_rm0 / no exec
    MGS_SEL_MGS1 | MGS_SEL_MGS2 , MGS_SEL_DMA , # load mask/ store mask
    0                           ,                # mas
    1                           ,                # next → slot 1
])



print('------ after slot init ------')
dfx_mng.print_debug()

## Step 8 — Inspect PR Region State

In [ ]:
dfx_man.grant_decoupler_to_ps()

for r in range(NUM_PR_REGION):
    dfx_man.release_decup(region=r)
    print(f'--- PR Region {r} ---')
    dfx_ip.get_pr_ctrl(r).print_status()

## Step 9 — Load All Partial Bitstreams into CMA

Allocate CMA buffers for every (region, rm) combination and register them
with the DFX Controller.

DFX Controller addressing:
- `slot_id` = region index
- `idx`     = rm index (trigger index within that region's VS)

In [ ]:
par_bs_bufs = {}   # (region, rm) → (buf, addr, size)

for r in range(NUM_PR_REGION):
    for m in range(NUM_RM_PER_REGION):
        buf, addr, size = dfx_ctrl.allocate_bit_stream_cma(PAR_BS[(r, m)])
        par_bs_bufs[(r, m)] = (buf, addr, size)
        print(f'  region {r} rm {m}: {PAR_BS[(r,m)]}  @ {hex(addr)}, size={hex(size)}')

In [ ]:
# Register each bitstream with the DFX Controller.
# slot_id = region index,  idx = rm index within that region.
for r in range(NUM_PR_REGION):
    for m in range(NUM_RM_PER_REGION):
        _, addr, size = par_bs_bufs[(r, m)]
        dfx_ctrl.set_simple_meta_data(r, m, addr, size)

print('--- DFX Controller status after config ---')
for r in range(NUM_PR_REGION):
    for m in range(NUM_RM_PER_REGION):
        dfx_ctrl.print_status(r)
        dfx_ctrl.print_simple_meta_data(r, m)

## Step 10 — Initial PR Trigger (Pre-load rm0 into Each Region)

Hold decouplers and trigger rm0 load for both regions so each region
starts with a valid bitstream before the DFX Manager takes control.
The DFX Manager will handle all subsequent reconfigurations autonomously
via the slot table (slots 0–4).

In [ ]:
for r in range(NUM_PR_REGION):
    dfx_man.hold_decup(region=r)

In [ ]:
for r in range(NUM_PR_REGION):
    print(f'Loading rm0 into region {r} ...')
    dfx_ctrl.trig(r, 0)             # slot_id=r, trigger_id=0 → rm0
    dfx_ctrl.restart_no_status(r)

for r in range(NUM_PR_REGION):
    dfx_ctrl.print_status(r)

In [ ]:
dfx_man.grant_decoupler_to_dfx_ctrl()

## Step 11 — Execute Pipeline and Wait for Interrupt

The DFX Manager sequences autonomously through the slot table:
```
0 (init: recon r0→rm0)
  → 1 (recon r1→rm0  + exec r0/rm0,  DMA in → MGS1)
  → 2 (recon r0→rm1  + exec r1/rm0,  MGS1   → MGS2)
  → 3 (recon r1→rm1  + exec r0/rm1,  MGS2   → MGS1)
  → 4 (recon r0→rm0  + exec r1/rm1,  MGS1   → DMA out)  ← interrupt fires here
  → 1 (cycles back for next batch)
```

In [ ]:
async def run_and_wait():
    start = time.perf_counter()
    dfx_mng.start_engine()
    await my_interrupt.wait()
    elapsed = time.perf_counter() - start
    print(f'Interrupt received. Elapsed: {elapsed:.6f} s')

loop = asyncio.get_event_loop()
loop.run_until_complete(run_and_wait())

In [ ]:
dfx_mng.print_debug()
dfx_mng.shutdown_engine()
dfx_mng.print_debug()

## Step 12 — Read Back and Verify Results

Each `Stream_Single_S2M` adds a compile-time constant `vm_id * amount_region + rm_id`:

| Stage       | Offset added |
|-------------|-------------|
| r0 / rm0    | 0×2 + 0 = **0** |
| r1 / rm0    | 1×2 + 0 = **2** |
| r0 / rm1    | 0×2 + 1 = **1** |
| r1 / rm1    | 1×2 + 1 = **3** |
| **Total**   | **6** |

Expected: `output == input_x + 6`

In [ ]:
buf_out.invalidate()
np_result = np.array(buf_out, dtype=np.float32).reshape(OUTPUT_SHAPE)

expected = np.load(os.path.join(PRJ_TC_DIR, 'y_pred_hls.npy'))
match    = np.array_equal(expected, np_result)

print('result   =', np_result.flatten()[:10], '...')
print('expected =', expected.flatten()[:10], '...')
print(f'\noutput == y_pred_hls: {match}')

if not match:
    diff = np_result.flatten() - expected.flatten()
    print('diff (first 10):', diff[:10])

## Step 13 — Per-Region Status (After Run)

In [ ]:
dfx_man.grant_decoupler_to_ps()

for r in range(NUM_PR_REGION):
    dfx_man.release_decup(region=r)
    print(f'--- PR Region {r} ---')
    dfx_ip.get_pr_ctrl(r).print_status()

## Step 14 — Debug Registers

In [ ]:
#dbg_val = overlay.mgs_debugger.read(0)

In [ ]:
# load_amount  = (dbg_val >>  0) & 0x7FF
# store_amount = (dbg_val >> 11) & 0x7FF
# state        = (dbg_val >> 22) & 0x1F
# print(f'dbg_val      = {hex(dbg_val)}')
# print(f'load_amount  = {load_amount}')
# print(f'store_amount = {store_amount}')
# print(f'state        = {state} (0x{state:02X})')

In [ ]:
print('mm2s_status')
print(dfx_dma.check_mm2s_status())

In [ ]:
print(dfx_dma.read(dfx_dma.MM2S_LENGTH))

In [ ]:
print(hex(dfx_dma.read(dfx_dma.MM2S_SA)))

In [ ]:
print('s2mm_status')
print(dfx_dma.check_s2mm_status())

In [ ]:
print(dfx_dma.read(dfx_dma.S2MM_LENGTH))

In [ ]:
print(hex(dfx_dma.read(dfx_dma.S2MM_DA)))